# Ciena MCP: explorando la API de OTDR

Este notebook documenta la investigación hecha sobre el módulo **OTDR**
(*Optical Time-Domain Reflectometer* — trazas de fibra) del Ciena MCP: cómo
se descubrió el módulo (no está documentado en el README ni en `mcp_client/`
original), qué endpoints expone, y qué se probó en vivo contra el MCP real.

**Estado:** exploración/documentación. La obtención efectiva de una traza
todavía no está resuelta (ver sección 5) — queda pendiente.


## 1. Cómo se descubrió el módulo `otdr`

El Ciena MCP expone un explorador Swagger propio en `/swagger-ui` (una SPA
Ember, no estático). Esa SPA carga su catálogo de módulos desde
`/nbis/nbis.json` — el mismo catálogo que usa el *API gateway* interno para
enrutar cada prefijo de URL (`/nsi`, `/pm`, `/otdr`, ...) al microservicio
que lo implementa.

Pedimos ese catálogo con el cliente ya autenticado:


In [ ]:
import requests, urllib3, json
from mcp_client import MCPClient

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

client = MCPClient()
token = client.auth.get_token()
headers = {"Authorization": f"Bearer {token}"}

r = requests.get(f"{client.base_url}/nbis/nbis.json", headers=headers, verify=False, timeout=15)
catalog = r.json()
print(f"{len(catalog)} interfaces registradas en el gateway\n")
print(json.dumps(catalog["otdr"], indent=2))


Dos detalles importantes de esta entrada:

- **`provider: submarine_7.2.15-2_0`**: el módulo OTDR lo implementa el
  microservicio de **submarine** (cable submarino), no un módulo genérico de
  óptica terrestre. Es posible que solo tenga datos útiles si la red incluye
  NEs submarinos con soporte OTDR.
- **`swagger-ui: /otdr/docs/api-docs/`**: el path del spec Swagger propio de
  este módulo (distinto del `/swagger-ui` general).
- **`publish: "False"`** en el `input`: no aparece listado en la navegación
  normal del `/swagger-ui` — por eso no lo habíamos visto antes. Aun así el
  gateway lo enruta (`output.publish: true`) y responde.


## 2. El spec Swagger de `otdr`

Con el path anterior pedimos el spec (Swagger 2.0) y listamos sus endpoints:


In [ ]:
r = requests.get(f"{client.base_url}/otdr/docs/api-docs/", headers=headers, verify=False, timeout=15)
spec = r.json()

print("swagger:", spec.get("swagger"), " basePath:", spec.get("basePath"), "\n")
for path, methods in spec["paths"].items():
    for method, info in methods.items():
        print(f"{method.upper():6} {path:45} {info.get('summary','')}")


### Resumen de los endpoints relevantes para obtener una traza en JSON

| Endpoint | Qué hace | Parámetro clave |
|---|---|---|
| `POST /otdr/api/v1/entities` | Inicia/detiene una traza sobre una entidad OTDR (ej. `OTDRCFG-1-5-8`) | requiere `session_id` de management |
| `GET /otdr/api/v1/entities` | Lista entidades OTDR disponibles en un NE | requiere `session_id` de management |
| `GET /otdr/api/v1/sortraces` | Lista trazas `.sor` **ya guardadas en el MCP** para un enlace | `fre_id` |
| `POST /otdr/api/v1/sortraces` | Dispara la recuperación de trazas `.sor` para un enlace | `fre_id` |
| `POST /otdr/api/v1/sor/parse` | Devuelve el **contenido parseado** (curva/eventos) de un `.sor` guardado, como JSON | `file_names: [...]` |
| `POST /otdr/api/v1/sorexport` | Exporta trazas `.sor` | — |
| `POST /otdr/api/v{1,2}/sordownload` | Envía el `.sor` crudo por SFTP a un perfil configurado | requiere `session_id` |

El campo `session_id` **no es un token cualquiera**: como se ve en la sección
5, identifica una sesión de *device management* activa contra un NE puntual,
manejada por un proxy interno (`RACTRL`). Todavía no localizamos qué
endpoint la crea.


## 3. Wrapper `OTDRService`

Siguiendo el mismo patrón que `NSIService`/`PMService`, se agregó
`mcp_client/otdr.py` con los métodos de lectura y el de lanzar/detener traza:


In [ ]:
from mcp_client import NSIService, OTDRService

nsi = NSIService(client)
otdr = OTDRService(client)

print([m for m in dir(otdr) if not m.startswith("_")])


## 4. Probando `saved_traces(fre_id)` contra datos reales

`saved_traces` es de solo lectura y no requiere `session_id`, así que se
puede probar directamente sobre los `fres` (enlaces) de un NE conocido:


In [ ]:
constructs = nsi.network_constructs()
lurin = next(c for c in constructs if c["attributes"].get("name") == "LURIN")
fres = nsi.facility_resources(lurin["id"])
print(f"NE={lurin['attributes']['name']}  {len(fres)} fres encontrados\n")

for fre in fres:
    traces = otdr.saved_traces(fre["id"])
    print(f"fre_id={fre['id']:<30} -> {len(traces)} trazas guardadas")


Ningún `fre` de LURIN tiene trazas guardadas. Para no descartar que sea solo
mala suerte con ese NE, se hizo un barrido acotado sobre una muestra de la
red completa — **con un límite duro de llamadas** para no generar carga
excesiva sobre el MCP de producción:


In [ ]:
call_budget = 150
calls_made = 0
found_any = False

types = {}
for nc in constructs:
    t = nc["attributes"].get("networkConstructType", "?")
    types[t] = types.get(t, 0) + 1
print(f"{len(constructs)} network constructs totales")
print("Tipos:", types, "\n")

for nc in constructs:
    if calls_made >= call_budget:
        print(f"Presupuesto de {call_budget} llamadas agotado, muestra parcial.")
        break
    fres_nc = nsi.facility_resources(nc["id"]); calls_made += 1
    for fre in fres_nc:
        if calls_made >= call_budget:
            break
        traces = otdr.saved_traces(fre["id"]); calls_made += 1
        if traces:
            found_any = True
            print(f"NE={nc['attributes'].get('name')} fre_id={fre['id']} -> {len(traces)} trazas")

print(f"calls_made={calls_made}")
if not found_any:
    print("No se encontraron trazas SOR guardadas en la muestra revisada.")


No se encontró ninguna traza guardada en la muestra (150 llamadas, sin
cubrir los 113 NEs completos). No es concluyente — falta ampliar el barrido,
o mejor, probar directo sobre un `fre_id` donde se sepa que ya se corrió una
traza OTDR manualmente desde la UI del MCP.


## 5. `entities()` y el problema del `session_id`

A diferencia de `saved_traces`, los endpoints para **listar entidades OTDR**
o **lanzar una traza nueva** (`GET`/`POST /otdr/api/v1/entities`) piden un
`session_id`. Probamos con un valor cualquiera para ver qué error devuelve:


In [ ]:
try:
    otdr.entities(session_id="test")
except Exception as e:
    resp = e.response
    print(resp.status_code, e)
    print(resp.text)


El error (`NEDISC-594`, vía proxy `RACTRL`) confirma que `session_id` **no**
es un token libre: identifica una **sesión de management activa contra ese
NE puntual**, gestionada por el proxy interno RACTRL. Esa sesión se
establece con algún otro endpoint que todavía no ubicamos en el catálogo
`nbis.json` (candidatos por nombre: `ractrl`, `internaltron`,
`commissioning`) — queda pendiente de investigar.

Sin esa sesión no se pueden listar entidades OTDR ni lanzar (`start`) una
traza nueva vía API.


## 6. Camino recomendado para obtener una traza en JSON

Con lo confirmado hasta ahora, si **ya existe** una traza `.sor` guardada
para un enlace conocido, el camino es directo y no requiere `session_id`:

```python
traces = otdr.saved_traces(fre_id)      # metadata: file_name, trace_type, timestamp, ...
file_name = traces[0]["attributes"]["file_name"]

parsed = otdr.parse_traces([file_name])  # {"data": [...], "results": {"success": [...], "failure": [...]}}
```

`parsed["data"]` trae el contenido ya parseado del archivo `.sor` (formato
estándar de trazas OTDR: curva de reflectancia, eventos, distancias) — eso
es lo que responde a la pregunta original de "traza OTDR en JSON".

Si en cambio se necesita **lanzar una traza nueva** porque no hay ninguna
guardada, falta resolver primero de dónde sale el `session_id` de
management (sección 5).


## Resumen y pendientes

| Método (`OTDRService`) | Endpoint | Probado en vivo | Resultado |
|---|---|---|---|
| `saved_traces(fre_id)` | `GET /otdr/api/v1/sortraces` | Sí | 200 OK, sin trazas en la muestra revisada |
| `entities(session_id)` | `GET /otdr/api/v1/entities` | Sí | 404 `NEDISC-594`: requiere sesión de management válida |
| `retrieve_traces(fre_id)` | `POST /otdr/api/v1/sortraces` | No | — |
| `parse_traces(file_names)` | `POST /otdr/api/v1/sor/parse` | No (sin `file_name` real disponible aún) | — |
| `start_stop_trace(...)` | `POST /otdr/api/v1/entities` | No | requiere `session_id` (ver arriba) |
| `tracelist(...)` | `GET /otdr/api/v1/tracelist` | No | requiere `session_id` |

**Pendiente:**
1. Encontrar un `fre_id` (o NE) donde ya exista una traza OTDR guardada, para
   probar `parse_traces` de punta a punta.
2. Ubicar el endpoint que entrega el `session_id` de management (probable
   candidato: módulo `ractrl` del catálogo `nbis.json`) para poder listar
   entidades OTDR y lanzar trazas nuevas.
